In [5]:
import sklearn
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [28]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)  # Disable line-wrapping
pd.set_option('display.max_rows', 20)

In [29]:
#coffee_clim_data = pd.read_csv(r"..\data\coll_caff_node_clim.csv")
#coffee_env_data = pd.read_csv(r"..\data\coll_caff_node_env.csv")
coffee_data = pd.read_csv(r"..\data\coll_all_new_jan.csv")
coffee_data_bin = pd.read_csv(r"..\data\coll_all_new_jan_bin.csv")

caffeine_content = pd.read_csv(r"..\input\no_caffeine_nodes_w_specimen.csv")

coffee_data.head

<bound method NDFrame.head of             specimen_id  source_crs  longitude   latitude                              mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded         specimen_name  clim_1_tmin1_jan  ...  env_73_asp  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old
0           abbayesii_0        4326  46.450000 -24.366660  POINT (647070.6204618097 7304410.220689219)              79              0                   False      Coffea_abbayesii             195.5  ...       307.0         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0
1           abbayesii_1        4326  46.830000 -24.730000  POINT (685086.8720605411 7263711.610572983)              79              0                   False      Coffea_abbayesii             177.5  ...        56.0         7630.0        10.0       

In [30]:
caffeine_content

,Species_name,caffeine_percent
0,C_andrambovatensis_A310,0.000
1,C_abbayesii_A601,0.000
2,C_arenesiana_A403,0.000
3,C_bertrandii_A5,0.000
4,C_dubardii_A969,0.000
...,...,...
21,C_vianneyi_A946,0.040
22,C_farafanganensis_A208,0.045
23,C_homollei_A945,0.060
24,C_kianjavatensis_A602,0.700


In [9]:
coffee_data.drop(coffee_data.columns[0], axis=1, inplace=True)
coffee_data.head

<bound method NDFrame.head of      source_crs  longitude   latitude                                mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded            specimen_name  clim_1_tmin1_jan  clim_2_tmin2_feb  ...  env_73_asp  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old
0          4326  46.450000 -24.366660    POINT (647070.6204618097 7304410.220689219)              79              0                   False         Coffea_abbayesii             195.5             195.0  ...       307.0         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0
1          4326  46.830000 -24.730000    POINT (685086.8720605411 7263711.610572983)              79              0                   False         Coffea_abbayesii             177.5             177.0  ...        56.0         7630.0        10

In [10]:
caffeine_content

,Species_name,caffeine_percent
0,C_andrambovatensis_A310,0.000
1,C_abbayesii_A601,0.000
2,C_arenesiana_A403,0.000
3,C_bertrandii_A5,0.000
4,C_dubardii_A969,0.000
5,C_heimii_A516,0.000
6,C_humbertii_RNF785,0.000
7,C_humblotiana_BM19_20,0.000
8,C_millotii_A222,0.000
9,C_perrieri_A12,0.000


In [31]:
# binary classification: 0 if caffeine_percent is 0 (or below), 1 if above 0
caffeine_content['caffeine_class'] = (caffeine_content['caffeine_percent'] > 0).astype(int)

print(caffeine_content)


               Species_name  caffeine_percent  caffeine_class
0   C_andrambovatensis_A310             0.000               0
1          C_abbayesii_A601             0.000               0
2         C_arenesiana_A403             0.000               0
3           C_bertrandii_A5             0.000               0
4           C_dubardii_A969             0.000               0
..                      ...               ...             ...
21          C_vianneyi_A946             0.040               1
22   C_farafanganensis_A208             0.045               1
23          C_homollei_A945             0.060               1
24    C_kianjavatensis_A602             0.700               1
25        C_lancifolia_A320             0.700               1

[26 rows x 3 columns]


In [12]:
coffee_data['extracted_specimen'] = coffee_data['specimen_name'].str.split('_').str[1]
print(coffee_data.head)

<bound method NDFrame.head of      source_crs  longitude   latitude                                mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded            specimen_name  clim_1_tmin1_jan  clim_2_tmin2_feb  ...  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old  extracted_specimen
0          4326  46.450000 -24.366660    POINT (647070.6204618097 7304410.220689219)              79              0                   False         Coffea_abbayesii             195.5             195.0  ...         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0           abbayesii
1          4326  46.830000 -24.730000    POINT (685086.8720605411 7263711.610572983)              79              0                   False         Coffea_abbayesii             177.5             177.0  ...         7630.0      

In [32]:
coffee_data_bin['extracted_specimen'] = coffee_data_bin['specimen_name'].str.split('_').str[1]
print(coffee_data_bin.head)

<bound method NDFrame.head of             specimen_id  source_crs  longitude   latitude                              mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded         specimen_name  clim_1_tmin1_jan  ...  env_78_wat_18  env_78_wat_15  env_78_wat_19  env_78_wat_24  env_78_wat_21  env_78_wat_0  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old  extracted_specimen
0           abbayesii_0        4326  46.450000 -24.366660  POINT (647070.6204618097 7304410.220689219)              79              0                    True      Coffea_abbayesii             195.5  ...              0              0              0              0              0             0                 159.0                  307.0                   30.0           abbayesii
1           abbayesii_1        4326  46.830000 -24.730000  POINT (685086.8720605411 7263711.610572983)              79              0                    True      Coffea_abbayesii             177.5  ...        

In [33]:
caffeine_content['extracted_species'] = caffeine_content['Species_name'].str.split('_').str[1]
print(caffeine_content)

               Species_name  caffeine_percent  caffeine_class extracted_species
0   C_andrambovatensis_A310             0.000               0  andrambovatensis
1          C_abbayesii_A601             0.000               0         abbayesii
2         C_arenesiana_A403             0.000               0        arenesiana
3           C_bertrandii_A5             0.000               0        bertrandii
4           C_dubardii_A969             0.000               0          dubardii
..                      ...               ...             ...               ...
21          C_vianneyi_A946             0.040               1          vianneyi
22   C_farafanganensis_A208             0.045               1   farafanganensis
23          C_homollei_A945             0.060               1          homollei
24    C_kianjavatensis_A602             0.700               1    kianjavatensis
25        C_lancifolia_A320             0.700               1        lancifolia

[26 rows x 4 columns]


In [14]:
merged_df = pd.merge(
    coffee_data,
    caffeine_content[['extracted_species', 'Species_name', 'caffeine_percent','caffeine_class']],
    left_on='extracted_specimen',
    right_on='extracted_species',
    how='left'  # Use 'left' join to keep all rows from coffee_env_data
)

# Step 4: Drop the helper columns
merged_df = merged_df.drop(columns=['extracted_specimen', 'extracted_species'])

# The result is the original DataFrame with an additional 'caffeine_class' column
merged_df.to_csv(r'../data/coll_caff_node_w_class.csv', index=False)

In [ ]:
merged_df

In [34]:
merged_df_bin = pd.merge(
    coffee_data_bin,
    caffeine_content[['extracted_species', 'Species_name', 'caffeine_percent','caffeine_class']],
    left_on='extracted_specimen',
    right_on='extracted_species',
    how='left'  # Use 'left' join to keep all rows from coffee_env_data
)

# Step 4: Drop the helper columns
merged_df_bin = merged_df_bin.drop(columns=['extracted_specimen', 'extracted_species'])

# The result is the original DataFrame with an additional 'caffeine_class' column
merged_df_bin.to_csv(r'../data/coll_caff_node_bin_w_class.csv', index=False)

In [17]:
input_df = merged_df[["Species_name","longitude","latitude","caffeine_percent"]]
input_df = input_df.rename(columns={'Species_name': 'specimen_id'})
input_df.to_csv("../data/coords_w_caff.csv", index=False)
input_df

,specimen_id,longitude,latitude,caffeine_percent
0,C_abbayesii_A601,46.450000,-24.366660,0.000
1,C_abbayesii_A601,46.830000,-24.730000,0.000
2,C_abbayesii_A601,46.833300,-24.733300,0.000
3,C_abbayesii_A601,46.833330,-24.666670,0.000
4,C_abbayesii_A601,46.833333,-24.733333,0.000
5,C_abbayesii_A601,46.850000,-24.800000,0.000
6,C_abbayesii_A601,46.863333,-24.760833,0.000
7,C_abbayesii_A601,46.962222,-24.836667,0.000
8,C_abbayesii_A601,47.136111,-24.842778,0.000
9,C_abbayesii_A601,47.136667,-24.835833,0.000
